# Build a Book Recommender using a Medallion Data Lake

This notebook takes the raw **Book-Crossing** dataset through a **Bronze → Silver → Gold** pipeline and then builds an **item-based collaborative filtering** recommender on top of the Gold layer.

```
raw CSV files ──► BRONZE (untouched) ──► SILVER (cleaned) ──► GOLD (model-ready) ──► recommender
```

- **Bronze**: exactly what the source contains, plus an ingestion timestamp.
- **Silver**: cleaned, validated, and filtered tables.
- **Gold**: a sparse rating matrix, lookup tables, and aggregates ready for modelling.

In [5]:
import os
import zipfile
from datetime import datetime

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.neighbors import NearestNeighbors

for folder in ["bronze", "silver", "gold"]:
    os.makedirs(folder, exist_ok=True)

SOURCE_ZIP = "/content/book-crossings.zip"

if not os.path.exists("BX-Users.csv"):
    if os.path.exists(SOURCE_ZIP):
        with zipfile.ZipFile(SOURCE_ZIP) as z:
            z.extractall(".")
    else:
        raise FileNotFoundError(f"{SOURCE_ZIP} not found. Please upload the dataset.")

print(sorted(f for f in os.listdir(".") if f.endswith(".csv")))

['BX-Book-Ratings.csv', 'BX-Books.csv', 'BX-Users.csv']


## Step 1: Bronze layer (raw copy)

The files use `;` as separator and are not UTF-8, so we read them with `encoding="latin-1"`.
We read every column as text (`dtype=str`) so nothing is converted or altered.
A few lines in `BX-Books.csv` have extra semicolons inside the title, so they cannot be parsed. We skip those lines at load time. This is a parsing issue, not a cleaning step.

In [6]:
def read_raw(filename):
    return pd.read_csv(
        filename,
        sep=";",
        encoding="latin-1",
        dtype=str,
        on_bad_lines="skip",
        low_memory=False,
    )

files = {"users": "BX-Users.csv", "books": "BX-Books.csv", "ratings": "BX-Book-Ratings.csv"}
ingestion_time = datetime.now().isoformat()

for name, filename in files.items():
    df = read_raw(filename)
    df["ingestion_timestamp"] = ingestion_time
    df.to_parquet(f"bronze/{name}.parquet", index=False)
    print(f"bronze/{name}.parquet  rows={len(df):,}  cols={list(df.columns)}")

bronze/users.parquet  rows=278,858  cols=['User-ID', 'Location', 'Age', 'ingestion_timestamp']
bronze/books.parquet  rows=271,360  cols=['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher', 'Image-URL-S', 'Image-URL-M', 'Image-URL-L', 'ingestion_timestamp']
bronze/ratings.parquet  rows=1,149,780  cols=['User-ID', 'ISBN', 'Book-Rating', 'ingestion_timestamp']


## Step 2: Explore the Bronze data (EDA)

We read the Bronze tables back from disk and check them before cleaning.

In [7]:
users_b = pd.read_parquet("bronze/users.parquet")
books_b = pd.read_parquet("bronze/books.parquet")
ratings_b = pd.read_parquet("bronze/ratings.parquet")

for name, df in [("users", users_b), ("books", books_b), ("ratings", ratings_b)]:
    print(f"--- {name} ---")
    print("shape:", df.shape)
    print(df.dtypes.to_string())
    print("missing values:\n" + df.isna().sum().to_string())
    print()

--- users ---
shape: (278858, 4)
User-ID                str
Location               str
Age                    str
ingestion_timestamp    str
missing values:
User-ID                     0
Location                    0
Age                    110762
ingestion_timestamp         0

--- books ---
shape: (271360, 9)
ISBN                   str
Book-Title             str
Book-Author            str
Year-Of-Publication    str
Publisher              str
Image-URL-S            str
Image-URL-M            str
Image-URL-L            str
ingestion_timestamp    str
missing values:
ISBN                   0
Book-Title             0
Book-Author            2
Year-Of-Publication    0
Publisher              2
Image-URL-S            0
Image-URL-M            0
Image-URL-L            3
ingestion_timestamp    0

--- ratings ---
shape: (1149780, 4)
User-ID                str
ISBN                   str
Book-Rating            str
ingestion_timestamp    str
missing values:
User-ID                0
ISBN               

In [8]:
ratings_num = pd.to_numeric(ratings_b["Book-Rating"], errors="coerce")
print("Rating value counts (0 = no explicit rating):")
print(ratings_num.value_counts().sort_index().to_string())
print("\nShare of zero ratings:", round((ratings_num == 0).mean() * 100, 1), "%")

Rating value counts (0 = no explicit rating):
Book-Rating
0     716109
1       1770
2       2759
3       5996
4       8904
5      50974
6      36924
7      76457
8     103736
9      67541
10     78610

Share of zero ratings: 62.3 %


In [9]:
ages = pd.to_numeric(users_b["Age"], errors="coerce")
print("Age summary:")
print(ages.describe().to_string())
print("Missing age:", ages.isna().sum())
print("Age < 5:", (ages < 5).sum(), "  Age > 100:", (ages > 100).sum())

years_raw = books_b["Year-Of-Publication"]
years = pd.to_numeric(years_raw, errors="coerce")
print("\nYear summary:")
print(years.describe().to_string())
print("Year == 0:", (years == 0).sum(), "  Year > 2026:", (years > 2026).sum())
print("Non-numeric year text:", years.isna().sum() - years_raw.isna().sum())
print(years_raw[years.isna() & years_raw.notna()].head(5).to_string())

Age summary:
count    168096.000000
mean         34.751434
std          14.428097
min           0.000000
25%          24.000000
50%          32.000000
75%          44.000000
max         244.000000
Missing age: 110762
Age < 5: 882   Age > 100: 366

Year summary:
count    271357.000000
mean       1959.760817
std         257.994226
min           0.000000
25%        1989.000000
50%        1995.000000
75%        2000.000000
max        2050.000000
Year == 0: 4618   Year > 2026: 11
Non-numeric year text: 3
209538    DK Publishing Inc
220731            Gallimard
221678    DK Publishing Inc


In [10]:
per_user = ratings_b["User-ID"].value_counts()
per_book = ratings_b["ISBN"].value_counts()
print("Ratings per user:")
print(per_user.describe().round(1).to_string())
print("\nRatings per book:")
print(per_book.describe().round(1).to_string())

n_pairs = len(users_b) * len(books_b)
n_rated_pairs = ratings_b[["User-ID", "ISBN"]].drop_duplicates().shape[0]
print(f"\nPossible user-book pairs: {n_pairs:,}")
print(f"Pairs with a rating:      {n_rated_pairs:,}")
print(f"Sparsity: {100 * (1 - n_rated_pairs / n_pairs):.4f} % of the matrix is empty")

Ratings per user:
count    105283.0
mean         10.9
std          90.6
min           1.0
25%           1.0
50%           1.0
75%           4.0
max       13602.0

Ratings per book:
count    340556.0
mean          3.4
std          12.4
min           1.0
25%           1.0
50%           1.0
75%           2.0
max        2502.0

Possible user-book pairs: 75,670,906,880
Pairs with a rating:      1,149,780
Sparsity: 99.9985 % of the matrix is empty


### EDA findings (one line each)

- **Shape, dtypes and missing values:** users 278,858 × 4, books 271,360 × 9, ratings 1,149,780 × 4. Every column loads as text. Missing values: 110,762 ages (about 40% of users), 2 book authors, 2 publishers and 3 large image URLs. In Silver I will convert the numeric columns, fill missing ages with the median, and drop the image URLs.
- **Rating distribution:** 716,109 ratings (62.3%) are 0, which means "no explicit rating". The 433,671 explicit ratings are skewed high: about 75% of them are 7 to 10. In Silver I will drop the zeros and keep ratings from 1 to 10.
- **User ages:** the range is 0 to 244, with a median of 32. There are 882 ages below 5 and 366 above 100, which are impossible values. In Silver I will set those to null and fill the nulls with the median of the cleaned ages.
- **Publication years:** 4,618 years are 0, 11 are after 2026 (the largest is 2050), and 3 contain publisher text such as "DK Publishing Inc" and "Gallimard". In Silver I will set 0 and future years to null, which also turns the text values into nulls.
- **Ratings per user and per book:** the median user has only 1 rating (75% have 4 or fewer, and the maximum is 13,602). The median book has 1 rating (75% have 2 or fewer, and the maximum is 2,502). In Silver I will keep only users and books with at least 10 ratings so the model has enough overlap to learn from.
- **Sparsity:** 99.9985% of user-book pairs are empty (1,149,780 rated pairs out of about 75.7 billion possible). This is why the Gold matrix is stored as a sparse matrix.

## Step 3: Silver layer (cleaning)

Each rule from the assignment is applied in order. Renaming the columns to `snake_case` makes the later code shorter.

In [11]:
CURRENT_YEAR = datetime.now().year
MIN_RATINGS = 10

users = pd.read_parquet("bronze/users.parquet")
books = pd.read_parquet("bronze/books.parquet")
ratings = pd.read_parquet("bronze/ratings.parquet")
print("bronze rows -> users:", len(users), " books:", len(books), " ratings:", len(ratings))

def tidy(s):
    return s.str.strip().str.replace(r"\s+", " ", regex=True)

# 1. Duplicates
users = users.drop_duplicates(subset=["User-ID"])
books = books.drop_duplicates(subset=["ISBN"])
ratings = ratings.drop_duplicates(subset=["User-ID", "ISBN"])

bronze rows -> users: 278858  books: 271360  ratings: 1149780


In [12]:
# 2. Users: age, and split Location into city / state / country
users = users.rename(columns={"User-ID": "user_id", "Location": "location", "Age": "age"})
users["user_id"] = users["user_id"].astype(int)

users["age"] = pd.to_numeric(users["age"], errors="coerce")
users.loc[(users["age"] < 5) | (users["age"] > 100), "age"] = np.nan
users["age"] = users["age"].fillna(users["age"].median())

loc = tidy(users["location"].fillna("").str.lower()).str.split(",", n=2, expand=True)
loc = loc.reindex(columns=[0, 1, 2])
users["city"] = loc[0].str.strip().replace("", np.nan)
users["state"] = loc[1].str.strip().replace("", np.nan)
users["country"] = loc[2].str.strip().replace("", np.nan)
users = users.drop(columns=["location"])

print("users after cleaning:", users.shape)
print(users[["user_id", "age", "city", "state", "country"]].head())

users after cleaning: (278858, 6)
   user_id   age         city            state         country
0        1  32.0          nyc         new york             usa
1        2  18.0     stockton       california             usa
2        3  32.0       moscow  yukon territory          russia
3        4  17.0        porto         v.n.gaia        portugal
4        5  32.0  farnborough            hants  united kingdom


In [13]:
# 3. Books: years, text cleaning, drop image URLs
books = books.rename(columns={
    "ISBN": "isbn", "Book-Title": "title", "Book-Author": "author",
    "Year-Of-Publication": "year", "Publisher": "publisher",
})
books = books.drop(columns=["Image-URL-S", "Image-URL-M", "Image-URL-L"])

books["isbn"] = books["isbn"].str.strip()
books["title"] = tidy(books["title"])
books["author"] = tidy(books["author"]).str.title()
books["publisher"] = tidy(books["publisher"])

books["year"] = pd.to_numeric(books["year"], errors="coerce")
books.loc[(books["year"] <= 0) | (books["year"] > CURRENT_YEAR), "year"] = np.nan

print("books after cleaning:", books.shape)
print(books[["isbn", "title", "author", "year"]].head())

books after cleaning: (271360, 6)
         isbn  ...    year
0  0195153448  ...  2002.0
1  0002005018  ...  2001.0
2  0060973129  ...  1991.0
3  0374157065  ...  1999.0
4  0393045218  ...  1999.0

[5 rows x 4 columns]


In [14]:
# 4. Ratings: keep 1 to 10, and keep only ratings whose user and book exist
ratings = ratings.rename(columns={"User-ID": "user_id", "ISBN": "isbn", "Book-Rating": "rating"})
ratings["isbn"] = ratings["isbn"].str.strip()
ratings["user_id"] = pd.to_numeric(ratings["user_id"], errors="coerce")
ratings["rating"] = pd.to_numeric(ratings["rating"], errors="coerce")

ratings = ratings[ratings["rating"].between(1, 10)]
ratings = ratings[ratings["isbn"].isin(books["isbn"]) & ratings["user_id"].isin(users["user_id"])].copy()
ratings["user_id"] = ratings["user_id"].astype(int)
ratings["rating"] = ratings["rating"].astype(int)

print("ratings after range and link checks:", len(ratings))

ratings after range and link checks: 383843


### Choosing the minimum number of ratings

The EDA showed that the median user and the median book each have only 1 rating. With a threshold that is too low, most users and books have too little overlap with others to produce a reliable similarity. With a threshold that is too high, most of the data is lost.

The table below shows how many ratings survive at each threshold, in a single pass. I chose **10**. At 5, the table still includes many users and books with only a handful of ratings. At 20 and 50, the number of books falls to about 2,100 and about 530, which throws away too much of the catalogue.

The final Silver numbers come from the repeating filter in the next cell. It removes a user or book, which can push others below the threshold, so it removes more than the single-pass table suggests. Its final result is 1,820 users, 2,030 books and 41,456 ratings.

In [15]:
u_cnt = ratings["user_id"].value_counts()
b_cnt = ratings["isbn"].value_counts()

for t in [5, 10, 20, 50]:
    kept = ratings[
        ratings["user_id"].isin(u_cnt[u_cnt >= t].index)
        & ratings["isbn"].isin(b_cnt[b_cnt >= t].index)
    ]
    print(f"threshold {t:>3}: users {(u_cnt >= t).sum():>6,}  books {(b_cnt >= t).sum():>6,}  ratings kept {len(kept):>8,}")

threshold   5: users 12,787  books 13,787  ratings kept  145,890
threshold  10: users  6,589  books  5,444  ratings kept   87,178
threshold  20: users  3,305  books  2,129  ratings kept   45,908
threshold  50: users  1,180  books    531  ratings kept   13,671


In [16]:
# 5. Keep users and books with at least MIN_RATINGS ratings.
# Removing one can push another below the threshold, so we repeat until nothing changes.
while True:
    before = len(ratings)
    u_cnt = ratings["user_id"].value_counts()
    b_cnt = ratings["isbn"].value_counts()
    ratings = ratings[
        ratings["user_id"].isin(u_cnt[u_cnt >= MIN_RATINGS].index)
        & ratings["isbn"].isin(b_cnt[b_cnt >= MIN_RATINGS].index)
    ]
    if len(ratings) == before:
        break

users = users[users["user_id"].isin(ratings["user_id"])]
books = books[books["isbn"].isin(ratings["isbn"])]

print(f"Final Silver -> users: {len(users):,}  books: {len(books):,}  ratings: {len(ratings):,}")

Final Silver -> users: 1,820  books: 2,030  ratings: 41,456


In [17]:
os.makedirs("silver", exist_ok=True)
users.to_parquet("silver/users.parquet", index=False)
books.to_parquet("silver/books.parquet", index=False)
ratings.to_parquet("silver/ratings.parquet", index=False)
print("Silver tables saved.")

Silver tables saved.


## Step 4: Gold layer (model-ready)

Gold contains:
- a **sparse user-book rating matrix** (most entries are empty, so a sparse format saves a lot of memory),
- a **book lookup** so recommendations can be shown as titles,
- **user-level and book-level aggregates** (average rating and number of ratings).

In [18]:
ratings = pd.read_parquet("silver/ratings.parquet")
books = pd.read_parquet("silver/books.parquet")
users = pd.read_parquet("silver/users.parquet")

user_cat = pd.Categorical(ratings["user_id"])
book_cat = pd.Categorical(ratings["isbn"])

rating_matrix = sp.csr_matrix(
    (ratings["rating"].astype(np.float32).values, (user_cat.codes, book_cat.codes)),
    shape=(len(user_cat.categories), len(book_cat.categories)),
)
sp.save_npz("gold/rating_matrix.npz", rating_matrix)

user_index = pd.DataFrame({"user_idx": np.arange(len(user_cat.categories)), "user_id": user_cat.categories})
book_index = pd.DataFrame({"book_idx": np.arange(len(book_cat.categories)), "isbn": book_cat.categories})

density = rating_matrix.nnz / (rating_matrix.shape[0] * rating_matrix.shape[1])
print("rating matrix (users x books):", rating_matrix.shape, " density:", round(density * 100, 4), "%")

rating matrix (users x books): (1820, 2030)  density: 1.1221 %


In [19]:
book_agg = ratings.groupby("isbn")["rating"].agg(avg_rating="mean", n_ratings="count").reset_index()
user_agg = ratings.groupby("user_id")["rating"].agg(avg_rating="mean", n_ratings="count").reset_index()
user_agg = user_agg.merge(users[["user_id", "age", "city", "state", "country"]], on="user_id", how="left")

book_lookup = book_index.merge(books[["isbn", "title", "author", "year"]], on="isbn", how="left")
book_lookup = book_lookup.merge(book_agg, on="isbn", how="left")
book_lookup["title_key"] = book_lookup["title"].str.lower()

user_index.to_parquet("gold/user_index.parquet", index=False)
book_index.to_parquet("gold/book_index.parquet", index=False)
book_lookup.to_parquet("gold/book_lookup.parquet", index=False)
user_agg.to_parquet("gold/user_aggregates.parquet", index=False)
book_agg.to_parquet("gold/book_aggregates.parquet", index=False)

print(book_lookup.head())
print("\nGold tables saved.")

   book_idx  ...                                          title_key
0         0  ...                             great gatsby (reissue)
1         1  ...                   lion, the witch and the wardrobe
2         2  ...  politically correct bedtime stories: modern ta...
3         3  ...  once upon a more enlightened time: more politi...
4         4  ...                  on a wicked dawn (cynster novels)

[5 rows x 8 columns]

Gold tables saved.


## Step 5: Item-based collaborative filtering

### The idea

Two books are similar if the **same users rated them** in a similar way. So we represent each book as a vector with one entry per user, and measure the angle between two book vectors with **cosine similarity**:

```
            user1  user2  user3  user4
Book A   [   9      0      8      0  ]
Book B   [   8      0      9      0  ]   → A and B point in almost the same direction → similar
Book C   [   0      7      0      9  ]   → points in a different direction → not similar
```

Cosine similarity is 1 when two vectors point the same way and 0 when they share no users. Because it looks only at the angle, it is not thrown off by books that have many more ratings than others.

To recommend for a user, we take the books they liked, find the books similar to each one, and score each candidate by `similarity × the user's rating` for the liked book. Books the user has already rated are removed.

In [20]:
rating_matrix = sp.load_npz("gold/rating_matrix.npz").tocsr()   # users x books
user_index = pd.read_parquet("gold/user_index.parquet")
book_lookup = pd.read_parquet("gold/book_lookup.parquet")

item_user = rating_matrix.T.tocsr()   # books x users: one row per book
print("item-user matrix (books x users):", item_user.shape)

knn = NearestNeighbors(metric="cosine", algorithm="brute")
knn.fit(item_user)
print("Model fitted.")

item-user matrix (books x users): (2030, 1820)
Model fitted.


In [21]:
def find_book_idx(title):
    key = title.strip().lower()
    hits = book_lookup[book_lookup["title_key"] == key]
    if hits.empty:
        hits = book_lookup[book_lookup["title_key"].str.contains(key, regex=False, na=False)]
    if hits.empty:
        raise ValueError(f"No book found with a title like '{title}'")
    return int(hits.sort_values("n_ratings", ascending=False).iloc[0]["book_idx"])

lookup = book_lookup.set_index("book_idx")

def similar_books(title, k=10):
    idx = find_book_idx(title)
    own_key = lookup.loc[idx, "title_key"]
    dist, nbrs = knn.kneighbors(item_user[idx], n_neighbors=5 * k + 1)

    picked, seen = [], {own_key}
    for j, d in zip(nbrs[0], dist[0]):
        key = lookup.loc[j, "title_key"]
        if j == idx or key in seen:
            continue
        seen.add(key)
        picked.append((j, 1 - d))
        if len(picked) == k:
            break

    out = lookup.loc[[j for j, _ in picked], ["title", "author"]].reset_index(drop=True)
    out["similarity"] = [round(float(s), 3) for _, s in picked]
    return out

In [22]:
def user_row(user_id):
    u = user_index.loc[user_index["user_id"] == user_id, "user_idx"]
    if u.empty:
        raise ValueError(f"User {user_id} is not in the Gold data")
    return rating_matrix[int(u.iloc[0])]

def liked_books(user_id, min_rating=7):
    row = user_row(user_id)
    keep = row.data >= min_rating
    out = lookup.loc[row.indices[keep], ["title", "author"]].reset_index(drop=True)
    out["rating"] = row.data[keep]
    out = out.sort_values("rating", ascending=False)
    return out.drop_duplicates(subset="title")

def recommend(user_id, n=10, like_threshold=7, k_neighbors=50):
    row = user_row(user_id)
    already_rated = set(row.indices)
    rated_titles = set(lookup.loc[row.indices, "title_key"])

    liked = row.data >= like_threshold
    seeds, seed_ratings = row.indices[liked], row.data[liked]
    if len(seeds) == 0:   # no high ratings: fall back to everything the user rated
        seeds, seed_ratings = row.indices, row.data

    dist, nbrs = knn.kneighbors(item_user[seeds], n_neighbors=k_neighbors + 1)

    scores = {}
    for seed, rating, d_row, n_row in zip(seeds, seed_ratings, dist, nbrs):
        for d, j in zip(d_row, n_row):
            if j == seed or j in already_rated:
                continue
            scores[j] = scores.get(j, 0.0) + (1 - d) * rating

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    picked, seen = [], set(rated_titles)
    for j, s in ranked:
        key = lookup.loc[j, "title_key"]
        if key in seen:
            continue
        seen.add(key)
        picked.append((j, s))
        if len(picked) == n:
            break

    out = lookup.loc[[j for j, _ in picked], ["title", "author"]].reset_index(drop=True)
    out["score"] = [round(float(s), 2) for _, s in picked]
    return out

## Step 6: Demo and sanity checks

First, check the item-based part: books most similar to the most-rated book in the dataset.

In [23]:
top_title = book_lookup.sort_values("n_ratings", ascending=False).iloc[0]["title"]
print(f"Most similar books to: '{top_title}'\n")
print(similar_books(top_title, k=10).to_string(index=False))

Most similar books to: 'The Lovely Bones: A Novel'

                                             title          author  similarity
Where the Heart Is (Oprah's Book Club (Paperback))    Billie Letts       0.219
                                  Lucky : A Memoir    Alice Sebold       0.218
                                 The Da Vinci Code       Dan Brown       0.178
                        The Pilot's Wife : A Novel    Anita Shreve       0.176
                                       Good in Bed Jennifer Weiner       0.176
                                Nights in Rodanthe Nicholas Sparks       0.170
                            The Pact: A Love Story    Jodi Picoult       0.168
                                    The Lake House James Patterson       0.166
                                             Lucky    Alice Sebold       0.165
      White Oleander : A Novel (Oprah's Book Club)     Janet Fitch       0.154


Next, run `recommend()` for three random users. For each one, the books they rated highly are shown next to the recommendations, so you can check whether the suggestions make sense.

In [24]:
user_counts = pd.Series(np.diff(rating_matrix.indptr), index=user_index["user_id"].values)
demo_users = user_counts.sample(3, random_state=1).index.tolist()

for uid in demo_users:
    print("=" * 70)
    print(f"User {uid}  (ratings given: {user_counts[uid]})")
    print("\nBooks this user rated highly (>= 7):")
    print(liked_books(uid, min_rating=7).head(8).to_string(index=False))
    print("\nRecommended books:")
    print(recommend(uid, n=10).to_string(index=False))
    print()

User 136491  (ratings given: 30)

Books this user rated highly (>= 7):
                                               title          author  rating
                             The Secret Life of Bees   Sue Monk Kidd    10.0
                              The Catcher in the Rye   J.D. Salinger    10.0
                  Plainsong (Vintage Contemporaries)      Kent Haruf    10.0
                                   The Da Vinci Code       Dan Brown    10.0
                  The Giver (21st Century Reference)      Lois Lowry    10.0
The Adventures of Huckleberry Finn (Bantam Classics)      Mark Twain    10.0
                           Girl with a Pearl Earring Tracy Chevalier    10.0
                                      A Time to Kill    John Grisham    10.0

Recommended books:
                                                           title              author  score
     The Tale of the Body Thief (Vampire Chronicles (Paperback))           Anne Rice  11.54
                    Memnoch the 

## Conclusion

We built a Bronze → Silver → Gold pipeline on the raw Book-Crossing data. The Gold layer stores a sparse rating matrix that feeds an item-based recommender. The recommender uses cosine similarity between books and ranks candidates by similarity weighted by the user's ratings.

Limitations: the dataset is very sparse and users rate few books, so the recommendations are only as good as the overlap between readers. Ratings of 0 were dropped, so the model does not learn from implicit signals such as books a user viewed but did not rate.